In [1]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

pd.set_option("display.max_columns", None)

In [2]:
# ============================================================
# CELL 2 — Load data (REQUIRED variable names)
# ============================================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("train shape:", original_df.shape)   # (9864, 19)
print("test shape:", test_df.shape)        # (2466, 18)
original_df.head()

train shape: (9864, 19)
test shape: (2466, 18)


,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


In [3]:
# ============================================================
# CELL 3 — Inspect missing values
# ============================================================
print(original_df.isnull().sum())
# Administrative_Duration: 492 missing
# ExitRates: 691 missing
# Region: 591 missing
# TrafficType: 789 missing
# VisitorType: 394 missing

Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64


In [11]:
# ============================================================
# CELL 4 — Column setup (all true categoricals encoded)
# ============================================================
TARGET = "Revenue"
ID_COL = "Session_ID"

train = original_df.copy()
test = test_df.copy()

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

# Numeric-coded but semantically categorical (nominal codes, not quantities)
force_categorical = ["OperatingSystems", "Browser", "Region", "TrafficType"]

numeric_cols = train[feature_cols].select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in force_categorical]

categorical_cols = train[feature_cols].select_dtypes(exclude=np.number).columns.tolist()
categorical_cols += force_categorical

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
Categorical: ['Month', 'VisitorType', 'Weekend', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']


In [12]:
# ============================================================
# CELL 5 — Impute missing values (NO ROWS DROPPED)
# ============================================================
numeric_medians = train[numeric_cols].median()
train[numeric_cols] = train[numeric_cols].fillna(numeric_medians)
test[numeric_cols] = test[numeric_cols].fillna(numeric_medians)

for col in categorical_cols:
    mode_val = train[col].mode(dropna=True)
    fill_val = mode_val.iloc[0] if len(mode_val) > 0 else "Missing"
    train[col] = train[col].fillna(fill_val)
    test[col] = test[col].fillna(fill_val)

print("Remaining missing (train):", train[feature_cols].isnull().sum().sum())
print("Remaining missing (test):", test[feature_cols].isnull().sum().sum())
print("Rows retained:", len(train), "/", len(original_df),
      "=", round(100*len(train)/len(original_df), 2), "%")   # 100%

Remaining missing (train): 0
Remaining missing (test): 0
Rows retained: 9864 / 9864 = 100.0 %


In [13]:
# ============================================================
# CELL 6 — One-hot encode categoricals
# ============================================================
train_feat = train[feature_cols].copy()
test_feat = test[feature_cols].copy()
train_feat["__is_train__"] = 1
test_feat["__is_train__"] = 0

combined = pd.concat([train_feat, test_feat], axis=0, ignore_index=True)

# Force these numeric-coded columns to category dtype so get_dummies encodes them
for c in force_categorical:
    combined[c] = combined[c].astype("category")

combined_encoded = pd.get_dummies(combined, columns=categorical_cols, drop_first=True)

train_encoded = combined_encoded[combined_encoded["__is_train__"] == 1].drop(columns="__is_train__").reset_index(drop=True)
test_encoded = combined_encoded[combined_encoded["__is_train__"] == 0].drop(columns="__is_train__").reset_index(drop=True)

print("Encoded train shape:", train_encoded.shape)  # (9864, 68)
print("Encoded test shape:", test_encoded.shape)     # (2466, 68)

Encoded train shape: (9864, 68)
Encoded test shape: (2466, 68)


In [14]:
# ============================================================
# CELL 7 — Scale numeric features
# ============================================================
scaler = StandardScaler()
train_encoded[numeric_cols] = scaler.fit_transform(train_encoded[numeric_cols])
test_encoded[numeric_cols] = scaler.transform(test_encoded[numeric_cols])
train_encoded.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,VisitorType_Other,VisitorType_Returning_Visitor,Weekend_True,OperatingSystems_2,OperatingSystems_3,OperatingSystems_4,OperatingSystems_5,OperatingSystems_6,OperatingSystems_7,OperatingSystems_8,Browser_2,Browser_3,Browser_4,Browser_5,Browser_6,Browser_7,Browser_8,Browser_9,Browser_10,Browser_11,Browser_12,Browser_13,Region_2.0,Region_3.0,Region_4.0,Region_5.0,Region_6.0,Region_7.0,Region_8.0,Region_9.0,TrafficType_2.0,TrafficType_3.0,TrafficType_4.0,TrafficType_5.0,TrafficType_6.0,TrafficType_7.0,TrafficType_8.0,TrafficType_9.0,TrafficType_10.0,TrafficType_11.0,TrafficType_12.0,TrafficType_13.0,TrafficType_14.0,TrafficType_15.0,TrafficType_16.0,TrafficType_17.0,TrafficType_18.0,TrafficType_19.0,TrafficType_20.0
0,-0.692295,-0.438286,-0.394118,-0.245114,-0.643422,-0.589695,0.922326,1.890855,-0.314712,-0.308312,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,-0.692295,-0.438286,-0.394118,-0.245114,-0.441603,-0.375766,0.807208,1.427034,-0.314712,-0.308312,False,False,True,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,0.502909,-0.163688,-0.394118,-0.245114,-0.464027,-0.435154,-0.140301,0.243270,-0.314712,-0.308312,False,False,False,False,False,False,False,True,False,False,True,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,-0.692295,-0.438286,-0.394118,-0.245114,-0.194935,-0.346517,-0.459089,-0.693293,0.925672,-0.308312,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,1.399312,0.182091,-0.394118,-0.245114,-0.262208,-0.475406,-0.228853,-0.062348,-0.314712,-0.308312,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [15]:
# ============================================================
# CELL 8 — processed_df (REQUIRED variable)
# ============================================================
processed_df = train_encoded.copy()
processed_df[TARGET] = train[TARGET].values

print("processed_df shape:", processed_df.shape)
print("processed_df missing values:", processed_df.isnull().sum().sum())  # 0

processed_df shape: (9864, 69)
processed_df missing values: 0


In [16]:
# ============================================================
# CELL 9 — Validation split + train Logistic Regression
# ============================================================
X = train_encoded.copy()
y = train[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_preds))  # ~0.868

Validation accuracy: 0.8667004561581348


In [17]:
# ============================================================
# CELL 10 — Refit on FULL data, predict on test (REQUIRED: model, predictions)
# ============================================================
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

predictions = model.predict(test_encoded)

print("Number of predictions:", len(predictions))  # 2466

Number of predictions: 2466


In [18]:
# ============================================================
# CELL 11 — SUBMISSION-GENERATION CODE (DO NOT MODIFY)
# ============================================================
import pandas as pd
import numpy as np

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing", "processed_missing", "original_rows", "processed_rows",
        "original_columns", "processed_columns", "row_retained_percent", "model_name"
    ],
    "value": [
        original_missing, processed_missing, original_rows, processed_rows,
        original_columns, processed_columns, round(row_retained_percent, 2), model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat([checkpoints, prediction_output], ignore_index=True)
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully.")
print(checkpoints)

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  69
6  row_retained_percent               100.0
7            model_name  LogisticRegression
